In [ ]:
# Import required modules
import numpy as np
import os
from pathlib import Path

# import pandas as pd
import skimage
import dask.array as da
import dask
from acid.utils.listdirNHF import listdirNHF
from acid.image_processing.rescale_intensity import quantize_image
from acid.feature_extraction.measure_haralick import (
    glcm_feature_map,
    haralick_compute_feature_count,
)

# boundary should be set to none for glcm feature map

# note the rescaling of the image intensities


In [2]:
from dask.distributed import Client

client = Client()
print(client)

<Client: 'tcp://127.0.0.1:40635' processes=6 threads=24, memory=61.91 GiB>


In [3]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 6
Total threads: 24,Total memory: 61.91 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40635,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:40689,Total threads: 4
Dashboard: http://127.0.0.1:39285/status,Memory: 10.32 GiB
Nanny: tcp://127.0.0.1:42405,


In [4]:
# input_dir_image = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\develop\260326_parallelization_strategy\fov_proc"
input_dir_image = Path.cwd().parent / "data" / "raw" / "proc" / "fov_proc"
# input_dir_mask = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\develop\260326_parallelization_strategy\seg"
input_dir_mask = Path.cwd().parent / "data" / "raw" / "proc" / "seg"
filenames = listdirNHF(Path(input_dir_image))

In [5]:
test_file_name = filenames[0]
test_file = skimage.io.imread(os.path.join(Path(input_dir_image), test_file_name))
print(test_file.shape)
print(test_file.dtype)
print(np.amax(test_file))
print(np.amin(test_file))
print(test_file.ndim)

(5, 1024, 1024)
float32
42736.656
22.040386
3


In [6]:
@dask.delayed
def load(filename, input_dir_image=input_dir_image, input_dir_mask=input_dir_mask):
    """
    needs to return an array. Expects filename to be the same for
    the image and segmentation mask.
    """
    image = skimage.io.imread(os.path.join(Path(input_dir_image), filename))
    mask = skimage.io.imread(os.path.join(Path(input_dir_mask), filename))

    return image, mask


@dask.delayed
def process_image_mask_mbfm(image, mask, input_channel_axis=-1, levels=8):
    """
    needs to receive an array and return an array.
    Expects the mask to be concatenated to the image data along the channel axis.

    This function is meant to prepare image and mask for the mask-based computation
    of the feature map, by concatenating the mask to the image data along the channel axis.
    """
    rescaled_image = []
    for c in np.unstack(image, axis=input_channel_axis):
        # introduce a very mild gaussian smoothing

        rescaled_channel = quantize_image(c, levels=levels)

        rescaled_image.append(rescaled_channel)

    rescaled_image = np.stack(rescaled_image, axis=input_channel_axis)
    mask = np.expand_dims(
        mask, axis=input_channel_axis
    )  # add a channel axis to the mask if it doesn't have one
    data = np.concatenate([rescaled_image, mask], axis=input_channel_axis)

    # ensure the correct order of the axes (channel axis in the last position)
    if input_channel_axis != -1:
        data = np.moveaxis(data, input_channel_axis, -1)

    return data, mask


@dask.delayed
def compute_mask_based_feature_map(
    data,
    feature_funct,
    feature_count_funct,
    depth=10,
    chunks=(256, 256, -1),
    boundary="reflect",
    dtype=float,
    trim=True,
    kwargs=None,
    window_shape_name="window_shape",
):
    """
    Compute a mask-restricted feature map over an n-dimensional image using chunked,
    overlap-aware processing via Dask.

    This function applies a user-defined feature extraction function (`feature_funct`)
    to spatial chunks of an input array, using `dask.array.map_overlap` to ensure
    correct computation at chunk boundaries. The computation is restricted to a region
    defined by a mask, which is expected to be concatenated as the last channel of the
    input array.

    The function supports 3-dimensional spatial data with an explicit channel axis,
    which is internally moved to the last position for consistency.

    Parameters
    ----------
    data : np.ndarray
        Input array of shape (..., C), where the last channel corresponds to a mask.
        The mask must be a single channel where 0 represents background and non-zero
        values represent the region of interest.

    feature_funct : callable
        A top-level function that computes a feature map from an image and a mask.
        It must have the signature:
            feature_funct(image=..., mask=..., **kwargs)
        and return an array with the same spatial dimensions as the input image (ake the same dimensions of
        input images minus the channel dimension or, in other words, the same dimensions of intput image
        minus 1),
        plus an additional feature dimension expected in position -1.

    feature_count_funct : callable
        A function that determines the number of output feature maps given the
        provided kwargs and number of channels. It must return:
            (F: int, stacked: bool)
        where F is the number of output features per pixel.

    depth : int or tuple of int, optional
        Overlap size for each spatial dimension when applying `map_overlap`.
        - If int: the same depth is applied to all spatial dimensions (aka all dimensions except the channel).
        - If tuple: must match the number of dimensions of `data`, with the last
        value (channel axis) equal to 0.

        When skimage.util.view_as_window function (or similar functions) is used for the pixel-by-pixel feature
        map computation, the depth should be at least half of the effective window size used by
        `feature_funct` to avoid border artifacts.

    chunks : tuple of int, optional
        Chunk size for each dimension of the input array. Must match the number
        of dimensions of `data`. The last dimension (channel axis) is typically
        not chunked (use -1).

    boundary : str, optional
        Boundary handling strategy for `dask.array.map_overlap`.
        Common options include 'reflect', 'nearest', or 'none'.

    dtype : data-type, optional
        Data type of the output array.

    trim : bool, optional
        Whether to trim overlapping regions after computation. Should generally
        be True to avoid duplicated computations at chunk borders.

    kwargs : dict, optional
        Additional keyword arguments passed to `feature_funct` and
        `feature_count_funct`.

        Must include:
        - 'channel_axis': int (cannot be None)

        May include:
        - window_shape (or equivalent), depending on `window_shape_name`

        NOTE: This dictionary is copied internally to avoid side effects.

    window_shape_name : str, optional
        Name of the keyword in `kwargs` corresponding to the window size used by
        `feature_funct` (e.g., 'window_shape').

        If provided, this is used to validate that the overlap (`depth`) is
        consistent with the window size to avoid edge artifacts.

    Returns
    -------
    np.ndarray
        Output feature map as a NumPy array. The shape matches the spatial
        dimensions of the input, with an additional feature dimension of size F.

    Notes
    -----
    - The last channel of `data` is assumed to be a mask and is not included in
    feature computation.
    - The function enforces the channel axis to be the last dimension internally.
    - Chunk sizes must be compatible with the specified overlap (`depth`), i.e.,
    depth must be smaller than chunk size along each spatial dimension.
    - This function executes eagerly and returns a NumPy array, even though it
    internally uses Dask for parallelization.

    Raises
    ------
    ValueError
        If invalid parameters are provided (e.g., mismatched dimensions, invalid
        depth, or channel_axis is None).

    AssertionError
        If assumptions about shapes, depth, or feature count are violated.


        needs to receive an array and return an array.
        Expects a multi-channel image.
        Expects channel axis in the last position.
        Expects the mask to be concatenated to the image data as the last sub-stack
        of the channel axis. This is required for chucking them together and passing the chucks as a dask object
        to map_overlap.

        feature_funct must be a top-level function that takes as input an image and a mask, and returns a feature map of the same spatial dimensions as the input image.
        It must accept a window_shape
        “window_shape_name must match the keyword expected by feature_funct”
        “channel_axis must not be None”
    """

    # define helper function to pass feature map computation to dask.map_overlap
    def _wrapper(data_block, **w_kwargs):
        # data_block is a numpy array corresponding to a chunk of the input data
        # it has the same structure as the input data (channel axis in the last position, mask concatenated to image data)

        img_block = data_block[
            ..., :-1
        ]  # all channels except the last one are image channels
        mask_block = data_block[..., -1]  # the last channel is the mask

        return feature_funct(image=img_block, mask=mask_block, **w_kwargs)

    # Use default kwargs
    if kwargs is None:
        kwargs = {"channel_axis": -1}
    else:
        kwargs = kwargs.copy()

    # enforce channel_axis to be in the last position
    ch_axis = kwargs.get("channel_axis", -1)
    if ch_axis is not None:
        if ch_axis != -1:
            data = np.moveaxis(data, ch_axis, -1)
            chunks = list(chunks)
            ch_chunk = chunks.pop(ch_axis)
            chunks.append(ch_chunk)
            chunks = tuple(chunks)

        kwargs["channel_axis"] = -1
    else:
        raise ValueError("channel_axis can't be None if passed to kwargs")

    # compute depth per each data dimension if depth is int
    if isinstance(depth, int):
        assert depth >= 1, "depth size must be positive"
        spatial_depth = [depth] * (data.ndim - 1)
        computed_depth = tuple(spatial_depth + [0])

    # if depth is tuple
    elif isinstance(depth, tuple):

        # assert that it matches the number of data dimension and that it is 0 on the last axis (consistency with channel axis)
        assert (
            len(depth) == data.ndim
        ), "if tuple depth must match the number of data dimensions"
        assert (
            depth[-1] == 0
        ), "Last depth dimension must be 0 (channel axis is not overlapped)"

        # if window_shape or other definition of the np.window shape is passed to kwargs
        # assert that the depth is always half of the window size in each dimension
        if window_shape_name in kwargs:
            ws = kwargs.get(window_shape_name)

            if isinstance(ws, int):
                for dp in depth[:-1]:
                    assert (
                        dp <= ws // 2
                    ), f"depth should match half of {window_shape_name} to avoid border artifacts"
            else:
                for d, w in zip(depth[:-1], ws):
                    assert (
                        d <= w // 2
                    ), f"depth should match half of {window_shape_name} in each dimension to avoid border artifacts"

        computed_depth = depth
    else:
        raise ValueError("depth must be an int or a tuple")

    # check that chunks have the same dimensions as input data
    assert len(chunks) == data.ndim, "chunks must match number of data dimensions"

    # check that depth is smaller than chunk size
    for d_, (c, dep) in enumerate(zip(chunks, computed_depth)):
        if d_ < len(chunks) - 1:  # ignore channel axis
            if c != -1 and dep >= c:
                raise ValueError(
                    f"Depth ({dep}) must be smaller than chunk size ({c}) in axis {d_}"
                )

    # determine the shape of the output chunks number of feature maps which are created
    n_channels = data.shape[-1] - 1
    F, stacked = feature_count_funct(kwargs, n_channels)
    assert isinstance(F, int) and F > 0, "Feature count must be a positive integer"
    out_chunks = (*chunks[:-1], F)

    # transfor data into dask array using specified chunk size
    # NOTE: as the mask is concatenated on the channel axis it is guaranteed the correct chunking of
    # image and mask.
    data_da = da.from_array(
        data, chunks=chunks
    )  # chunks in xy but not on the channel axis

    # map feature map computation over data chunks, with overlap
    fm = data_da.map_overlap(
        _wrapper,
        depth=computed_depth,
        boundary=boundary,
        dtype=dtype,
        trim=trim,  # trim the overlapping regions after computation to avoid double-counting
        chunks=out_chunks,
        **kwargs,
    )

    # compute feature map over chunks
    return fm.compute()  # returns a numpy array


@dask.delayed
def measure_feature_map(feature_map, mask):
    # generates measurements from the feature map and the mask
    return  # returns a dataframe of measurements


def f_mbfm(
    filenames,
    feature_funct,
    feature_count_funct,
    channel_axis=-1,
    levels=8,
    window_shape=11,
    chunks=(256, 256, -1),
    boundary="reflect",
    dtype=float,
    trim=True,
    kwargs=None,
):
    """ """
    if kwargs is None:
        kwargs = {}

    results = []

    for filename in filenames:

        image_mask = load(filename)

        image = image_mask[0]
        mask = image_mask[1]

        data_mask = process_image_mask_mbfm(
            image, mask, input_channel_axis=channel_axis, levels=levels
        )

        data = data_mask[0]
        mask = data_mask[1]

        fm = compute_mask_based_feature_map(
            data,
            feature_funct,
            feature_count_funct,
            # window_shape=window_shape,
            chunks=chunks,
            boundary=boundary,
            dtype=dtype,
            trim=trim,
            kwargs=kwargs,
        )
        print(fm.shape)
        measurements = measure_feature_map(fm, mask)

        results.append(measurements)

    return results

In [7]:
channel_axis = 0
levels = 8
window_shape = 11
chunks = (
    256,
    256,
    -1,
)  # channel axis is moved to the last position after preprocessing
boundary = "none"  # none is used for haralick features because the image (and image chunks) are already padded within glcm_feature_map_ch
proc_dtype = float
trim = True
glcm_kwargs = {
    "props": None,
    "distances": None,
    "angles": None,
    "graycomtx_kwargs": {"levels": levels, "symmetric": True, "normed": True},
    "pad_kwargs": None,
    "windows_kwargs": None,
    "zeros_kwargs": None,
    "glcm_concat_kwargs": None,
    "stack_channels": False,  # must be False for dask.map_overlap to work
    "stack_axis": -1,
    "stack_kwargs": None,
    "feature_concat_axis": -1,  # must be in the same position of the channel axis, which has been move to the last position after preprocessing.
    "feature_concat_kwargs": None,
}

In [8]:
dask.compute(
    f_mbfm(
        filenames,
        feature_funct=glcm_feature_map,
        feature_count_funct=haralick_compute_feature_count,
        channel_axis=channel_axis,
        levels=levels,
        window_shape=window_shape,
        chunks=chunks,
        boundary=boundary,
        dtype=proc_dtype,
        trim=trim,
        kwargs=glcm_kwargs,
    )
)

Delayed('getattr-9e648124e53feddf9d2e6ee60ea20d8c')
Delayed('getattr-f25bf0cff7cc7125d879b9dfa6bb4d32')
Delayed('getattr-a724e7f99f98d988d1abb24533e01f7b')
Delayed('getattr-f986da8389976fd2e889ff48742487fc')
Delayed('getattr-8ef819322041d25e8ce02147e803e53b')
Delayed('getattr-c2a7d5f12cbfae36676c9bfca15ff418')
Delayed('getattr-f994e010d43215fc7ffbee61de03877e')
Delayed('getattr-85cfb09288ab5a267e3aaa7de563ba96')
Delayed('getattr-6bfb4dd19cf025b87c4e76334aec353f')
Delayed('getattr-98b3912a7215ee39745985d01108b4dd')
Delayed('getattr-2f2a07a1b480c6f4478a5ab1d97f202e')
Delayed('getattr-655b15bfeca6cfde831dc61de1f79843')
Delayed('getattr-df7d344919fd1fc80c6787fc3950fe3f')
Delayed('getattr-edc6dc193d3de72386cb0204e99cf2b0')
Delayed('getattr-8339c770e137b1ae9b8b98ab884b3b13')
Delayed('getattr-026d3e1d121006b68ab9238be7435f97')
Delayed('getattr-3540eadc218be78a8c2fe9e1301cd48b')
Delayed('getattr-6489134d49e828391071963a07cbba8e')
Delayed('getattr-220a95f9e949d61a1135c8298d53acb9')
Delayed('get

/home/christine/Sandbox/ACID/.venv/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 12.00 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/home/christine/Sandbox/ACID/.venv/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 12.00 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/home/christine/Sandbox/ACID/.venv/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 12.00 MiB

([None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,


In [9]:
# # Indicate the path to the input image
# input_image___path = os.path.join(os.getcwd(),"secondary_output")

# # get the name of the input image
# input_image___name = "260202_fov_example.ome.tif"

# # open input image
# input_real_image = io.imread(os.path.join(input_image___path, input_image___name))
# print(input_real_image.shape)

# # the first image of the channel axis is the segmentation mask. The following images are channels, to be analysed

the file is a 2720x2720 field of view with 7 channels.

The first of the 7 channels is the segmentation mask.

Objects are individual cells.

The remaining 6 channels are different imaged structures / imaging modalities.

In [10]:
# real_data_haralick_features = measure_haralick_features(image=input_real_image[...,1:3],
#                                              label_image=input_real_image[...,0],
#                                              props=['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM'],
#                                              distances=[5, 15, 49],
#                                              angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
#                                              channel_axis=-1,
#                                              window_shape=50,
#                                              glcm_daskbag_kwargs={'npartitions': 10})

# real_data_haralick_features

# # single channel,, window_shape==50, 6 prop, 3 distance [5, 15, 49], 4 angles [0, np.pi/4, np.pi/2, 3*np.pi/4], 10 partitions -> 8m 16s
# # double channels,, window_shape==50, 6 prop, 3 distance [5, 15, 49], 4 angles [0, np.pi/4, np.pi/2, 3*np.pi/4], 12 partitions -> 15m 33s